# 02 · Tool Use
### Practical LLM Engineering — Module 05: LLM Agents

> **Objectives**
> - Building a rich, extensible tool library with a consistent interface
> - Code execution: safe Python sandbox with output capture
> - Data tools: CSV analysis, unit conversion, web search
> - Tool chaining: outputs of one tool feed the next
> - Timeout, sandboxing, and output-size guards for production use
> - Observability: tool call traces and latency breakdown

---


## 1. Overview

A **tool library** is the practical foundation of any LLM agent. Agent capability is bounded by tool quality:

```
Agent capability ≈ f(reasoning_quality, tool_quality, tool_coverage)
```

### Tool risk levels

| Category | Examples | Risk |
|---|---|---|
| **Compute** | calculator, code runner | Low–High |
| **Read** | search, file read, DB query | Low–Medium |
| **Write** | file write, DB update | High |
| **Communicate** | email, Slack, SMS | High |
| **Execute** | shell, subprocess | Very High |



## 2. Setup

In [ ]:
# !pip install numpy matplotlib -q
import os, re, json, time, math, random, textwrap, io, contextlib, threading
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, Any, Callable
from collections import defaultdict
import math as _math

@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, system, user, tools=None, max_tokens=512, temperature=0.0):
        time.sleep(0.02)
        text = f"Based on the context: {user[:60]}... Here is the analysis and result."
        return LLMResponse(text, len((system+user).split()), len(text.split()), 40.0)
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

llm = MockLLM()

@dataclass
class ToolResult:
    tool: str; success: bool; output: Any; error: Optional[str]; latency_ms: float

class Tool(ABC):
    name: str; description: str; schema: dict
    @abstractmethod
    def run(self, **kwargs) -> ToolResult: ...
    def to_schema(self): return {"name": self.name, "description": self.description,
                                  "input_schema": self.schema}
    def _ok(self, output, ms=0.0):  return ToolResult(self.name, True,  output, None, ms)
    def _err(self, error, ms=0.0):  return ToolResult(self.name, False, None,  error, ms)

print("✅ Shared utilities ready")


## 3. Tool Library Implementation

In [ ]:
# ── Tool 1: Python Sandbox ───────────────────────────────────────────
class PythonSandbox(Tool):
    name        = "python_exec"
    description = "Execute Python code and return printed output. For calculations and data analysis."
    schema      = {"type": "object",
                   "properties": {"code": {"type": "string", "description": "Python code to run"}},
                   "required": ["code"]}

    SAFE = {"print": print, "len": len, "range": range, "enumerate": enumerate,
            "zip": zip, "sorted": sorted, "sum": sum, "min": min, "max": max,
            "abs": abs, "round": round, "int": int, "float": float, "str": str,
            "list": list, "dict": dict, "set": set, "type": type,
            "math": _math, "json": json}

    def run(self, code):
        t0, buf = time.perf_counter(), io.StringIO()
        try:
            with contextlib.redirect_stdout(buf):
                exec(code, {"__builtins__": self.SAFE})
            output = buf.getvalue().strip() or "[no output]"
            return self._ok(output, (time.perf_counter()-t0)*1000)
        except Exception as e:
            return self._err(f"{type(e).__name__}: {e}", (time.perf_counter()-t0)*1000)


# ── Tool 2: Calculator ────────────────────────────────────────────────
class Calculator(Tool):
    name        = "calculator"
    description = "Evaluate a mathematical expression. Supports: +,-,*,/,**,sqrt,sin,cos,log,pi,e."
    schema      = {"type": "object",
                   "properties": {"expression": {"type": "string"}},
                   "required": ["expression"]}
    _SAFE = {k: getattr(_math,k) for k in dir(_math) if not k.startswith("_")}
    _SAFE.update({"abs":abs,"round":round,"min":min,"max":max,"sum":sum})

    def run(self, expression):
        t0 = time.perf_counter()
        try:
            result = eval(expression, {"__builtins__": {}}, self._SAFE)
            return self._ok({"expression": expression, "result": result},
                             (time.perf_counter()-t0)*1000)
        except Exception as e:
            return self._err(str(e), (time.perf_counter()-t0)*1000)


# ── Tool 3: Unit Converter ────────────────────────────────────────────
class UnitConverter(Tool):
    name        = "unit_convert"
    description = "Convert a value between units (temperature, length, weight, speed)."
    schema      = {"type": "object",
                   "properties": {
                       "value":     {"type": "number"},
                       "from_unit": {"type": "string"},
                       "to_unit":   {"type": "string"}},
                   "required": ["value", "from_unit", "to_unit"]}
    _TABLE = {
        ("km","m"):1000, ("m","km"):0.001, ("miles","km"):1.60934, ("km","miles"):0.621371,
        ("feet","m"):0.3048, ("m","feet"):3.28084, ("inches","cm"):2.54, ("cm","inches"):0.3937,
        ("lb","kg"):0.453592, ("kg","lb"):2.20462, ("oz","g"):28.3495, ("g","oz"):0.035274,
        ("kmh","mph"):0.621371, ("mph","kmh"):1.60934, ("ms","kmh"):3.6, ("kmh","ms"):0.277778,
    }
    def _temp(self, v, frm, to):
        to_c = {"celsius":v, "fahrenheit":(v-32)*5/9, "kelvin":v-273.15}
        fr_c = {"celsius":lambda c:c, "fahrenheit":lambda c:c*9/5+32, "kelvin":lambda c:c+273.15}
        return fr_c[to](to_c[frm]) if frm in to_c and to in fr_c else None

    def run(self, value, from_unit, to_unit):
        f, t = from_unit.lower(), to_unit.lower()
        result = self._temp(value, f, t) if f in {"celsius","fahrenheit","kelvin"}                  else (value * self._TABLE.get((f, t))) if (f, t) in self._TABLE else None
        if result is None:
            return self._err(f"No conversion: {from_unit} → {to_unit}")
        return self._ok({"value": value, "from": from_unit, "to": to_unit,
                          "result": round(result, 6)})


# ── Tool 4: CSV Analyser ──────────────────────────────────────────────
class CSVAnalyser(Tool):
    name        = "analyse_csv"
    description = "Analyse a CSV dataset: compute summary stats, describe columns, check missing values."
    schema      = {"type": "object",
                   "properties": {
                       "csv_data":  {"type": "string"},
                       "operation": {"type": "string", "enum": ["summary","describe","head","missing"]}},
                   "required": ["csv_data", "operation"]}

    def run(self, csv_data, operation="summary"):
        import csv as _csv
        try:
            reader = _csv.DictReader(io.StringIO(csv_data))
            rows   = list(reader)
            if not rows: return self._err("Empty CSV")
            cols   = list(rows[0].keys())
            if operation == "head":
                return self._ok({"columns": cols, "rows": rows[:3]})
            if operation == "missing":
                return self._ok({"missing": {c: sum(1 for r in rows if not r.get(c)) for c in cols},
                                  "total_rows": len(rows)})
            if operation == "describe":
                stats = {}
                for col in cols:
                    vals = []
                    for r in rows:
                        try: vals.append(float(r[col]))
                        except: pass
                    if vals:
                        stats[col] = {"min": min(vals), "max": max(vals),
                                       "mean": round(sum(vals)/len(vals), 3), "n": len(vals)}
                return self._ok(stats)
            # summary
            return self._ok({"n_rows": len(rows), "n_cols": len(cols),
                               "columns": cols, "sample": rows[0]})
        except Exception as e:
            return self._err(str(e))


# ── Tool 5: Web Search ────────────────────────────────────────────────
class WebSearch(Tool):
    name        = "web_search"
    description = "Search the web for information on a topic. Returns titles and snippets."
    schema      = {"type": "object",
                   "properties": {
                       "query":       {"type": "string"},
                       "max_results": {"type": "integer", "default": 3}},
                   "required": ["query"]}

    def run(self, query, max_results=3):
        return self._ok({"query": query, "results": [
            {"rank": i+1,
             "title": f"{'Introduction to' if i==0 else 'Advanced'} {query}",
             "snippet": f"This resource covers {query} with practical examples and theory.",
             "url": f"https://docs.example.com/{query.replace(' ','-')}/{i+1}"}
            for i in range(min(max_results, 5))
        ]})


# ── Register ──────────────────────────────────────────────────────────
TOOL_LIBRARY = {
    "python_exec":  PythonSandbox(),
    "calculator":   Calculator(),
    "unit_convert": UnitConverter(),
    "analyse_csv":  CSVAnalyser(),
    "web_search":   WebSearch(),
}

print("Tool library:")
for name, tool in TOOL_LIBRARY.items():
    print(f"  {name:<16} — {tool.description[:65]}")


## 4. Exercising the Tool Library

In [ ]:
# ── Python sandbox ───────────────────────────────────────────────────
sandbox = TOOL_LIBRARY["python_exec"]
print("Python Sandbox:\n")
for snippet in [
    "fib = lambda n: n if n<=1 else fib(n-1)+fib(n-2)\nprint([fib(i) for i in range(10)])",
    "import math\nprint([round(math.sin(math.pi*i/4),3) for i in range(8)])",
    "print(1/0)   # error case",
]:
    r = sandbox.run(code=snippet)
    s = "✅" if r.success else "❌"
    print(f"  {s} {snippet.splitlines()[0][:45]}")
    print(f"     → {str(r.output or r.error)[:70]}\n")

# ── Calculator ────────────────────────────────────────────────────────
calc = TOOL_LIBRARY["calculator"]
print("Calculator:\n")
for expr in ["sqrt(2025)", "log(e**3)", "factorial(10)", "sin(pi/2)**2 + cos(pi/2)**2"]:
    r = calc.run(expression=expr)
    print(f"  {expr:<35} = {r.output['result'] if r.success else r.error}")

# ── Unit converter ────────────────────────────────────────────────────
conv = TOOL_LIBRARY["unit_convert"]
print("\nUnit Converter:\n")
for val, frm, to in [(100,"celsius","fahrenheit"),(26.2,"miles","km"),(70,"kg","lb"),(120,"kmh","mph")]:
    r = conv.run(value=val, from_unit=frm, to_unit=to)
    if r.success:
        print(f"  {val} {frm:<14} → {r.output['result']:.3f} {to}")

# ── CSV analyser ──────────────────────────────────────────────────────
CSV_DATA = "name,age,score,grade\nAlice,28,92.5,A\nBob,34,78.0,B\nCarol,25,88.5,B+\nDave,30,95.0,A+\nEve,22,71.5,C+"
csv_tool = TOOL_LIBRARY["analyse_csv"]
print("\nCSV Analysis:\n")
for op in ["summary", "describe", "missing"]:
    r = csv_tool.run(csv_data=CSV_DATA, operation=op)
    print(f"  [{op}]  {json.dumps(r.output, default=str)[:90]}")


## 5. Tool Chaining

In [ ]:
class ToolChain:
    """Execute an ordered sequence of tool calls; thread outputs through context."""
    def __init__(self, library):
        self.library = library
        self.history = []
        self._ctx    = {}   # accumulated key-value context

    def step(self, tool_name, **kwargs):
        # Substitute {key} placeholders from context
        kwargs = {k: (v.format(**self._ctx) if isinstance(v, str) else v)
                  for k, v in kwargs.items()}
        if tool_name not in self.library:
            return ToolResult(tool_name, False, None, f"Unknown: {tool_name}", 0.0)
        r = self.library[tool_name].run(**kwargs)
        self.history.append({"step": len(self.history)+1, "tool": tool_name,
                               "ok": r.success, "output": r.output})
        if r.success and isinstance(r.output, dict):
            for k, v in r.output.items():
                self._ctx[f"step{len(self.history)}_{k}"] = v
        elif r.success:
            self._ctx[f"step{len(self.history)}_output"] = r.output
        return r

    def summary(self):
        print("Chain summary:")
        for h in self.history:
            s = "✅" if h["ok"] else "❌"
            print(f"  {h['step']}. {s} {h['tool']:16} → {str(h['output'])[:60]}")


CSV_DATA = "name,age,score,grade\nAlice,28,92.5,A\nBob,34,78.0,B\nCarol,25,88.5,B+\nDave,30,95.0,A+\nEve,22,71.5,C+"

# Chain: CSV summary → calculate stats → run analysis code
chain = ToolChain(TOOL_LIBRARY)

print("Tool chain: CSV → stats → Python analysis\n")
r1 = chain.step("analyse_csv", csv_data=CSV_DATA, operation="describe")
if r1.success and "score" in r1.output:
    score_stats = r1.output["score"]
    r2 = chain.step("calculator",
                     expression=f"{score_stats['max']} - {score_stats['min']}")
    r3 = chain.step("python_exec", code=(
        "scores = [92.5, 78.0, 88.5, 95.0, 71.5]\n"
        "mean = sum(scores)/len(scores)\n"
        "print(f'mean={mean:.1f}, above_mean={sum(1 for s in scores if s>=mean)}')"))

chain.summary()


## 6. Guarded Tool Wrapper

In [ ]:
class GuardedTool:
    """Wraps any Tool with timeout + output size limits."""
    def __init__(self, tool, max_output_chars=2000, timeout_s=10.0):
        self.tool             = tool
        self.max_output_chars = max_output_chars
        self.timeout_s        = timeout_s
        self.name             = tool.name

    def run(self, **kwargs):
        box = [None]
        def target(): box[0] = self.tool.run(**kwargs)
        t0 = time.perf_counter()
        th = threading.Thread(target=target, daemon=True)
        th.start(); th.join(timeout=self.timeout_s)
        ms = (time.perf_counter()-t0)*1000
        if th.is_alive():
            return ToolResult(self.tool.name, False, None, f"Timeout after {self.timeout_s}s", ms)
        r = box[0]
        if r and r.success and isinstance(r.output, str) and len(r.output) > self.max_output_chars:
            r.output = r.output[:self.max_output_chars] + f"\n[truncated at {self.max_output_chars} chars]"
        return r


# Benchmark guarded vs unguarded
guarded_sandbox = GuardedTool(PythonSandbox(), max_output_chars=200, timeout_s=2.0)

print("Guarded tool tests:\n")
tests = [
    ("Fast computation",  "print(sum(range(1000)))"),
    ("Large output",      "for i in range(50): print(i)"),
    ("ZeroDivisionError", "print(1/0)"),
]
for label, snippet in tests:
    r = guarded_sandbox.run(code=snippet)
    s = "✅" if r.success else "❌"
    print(f"  {s} [{label}]")
    if r.output: print(f"     output: {str(r.output)[:80]}")
    if r.error:  print(f"     error:  {r.error[:60]}")
    print()


## 7. Tool Call Observability

In [ ]:
class ObservableToolLibrary:
    """Wraps a tool library to record latency, success rates, and call counts."""
    def __init__(self, library):
        self._library = library
        self._metrics = defaultdict(lambda: {"calls":0, "ok":0, "total_ms":0.0})

    def call(self, tool_name, **kwargs):
        if tool_name not in self._library:
            return ToolResult(tool_name, False, None, f"Unknown: {tool_name}", 0.0)
        t0 = time.perf_counter(); r = self._library[tool_name].run(**kwargs)
        ms = (time.perf_counter()-t0)*1000
        m  = self._metrics[tool_name]
        m["calls"] += 1; m["ok"] += int(r.success); m["total_ms"] += ms
        return r

    def report(self):
        print(f"\n{'Tool':<16} {'Calls':>6} {'Success%':>9} {'Avg ms':>8}")
        print("-"*44)
        for name, m in self._metrics.items():
            avg = m["total_ms"]/m["calls"] if m["calls"] else 0
            pct = 100*m["ok"]/m["calls"]   if m["calls"] else 0
            print(f"{name:<16} {m['calls']:>6} {pct:>8.0f}% {avg:>8.1f}")


obs = ObservableToolLibrary(TOOL_LIBRARY)

# Run a variety of calls
for _ in range(3): obs.call("calculator",   expression="sqrt(144)")
for _ in range(2): obs.call("unit_convert",  value=100, from_unit="celsius", to_unit="fahrenheit")
for _ in range(4): obs.call("web_search",    query="HNSW algorithm")
obs.call("calculator", expression="1/0")           # error case
obs.call("python_exec", code="print('hello')")

obs.report()


## 8. Key Takeaways

| Concept | Summary |
|---|---|
| **Tool class** | Abstract base with name, description, schema, run() |
| **Safe sandbox** | Restrict builtins; capture stdout; never allow imports of os/sys |
| **Tool chaining** | Context dict threads outputs between steps |
| **Guarded wrapper** | Timeout + output size limit for every production tool |
| **Observability** | Track call count, success rate, latency per tool |
| **Risk levels** | Compute < read < write < execute < communicate |


